In [116]:
# Imports 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path 

# Torch imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [117]:
# Select the device 

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [118]:
# Load the dataset paths (training, testing)RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x128 and 64x10)

train_data_path = Path('../datasets/cifar10/train')
test_data_path = Path('../datasets/cifar10/test')

In [119]:
# Get all the training and testing images

train_data = []
test_data = []

train_labels = []
test_labels = []

class_to_idx = {
    "airplane": 0,
    "automobile": 1,
    "bird": 2,
    "cat": 3,
    "deer": 4,
    "dog": 5,
    "frog": 6,
    "horse": 7,
    "ship": 8,
    "truck": 9,
}

for _class in train_data_path.iterdir():
    for img in _class.iterdir():
        _img = Image.open(img)
        grey = _img.convert("L")

        train_data.append(np.array(grey))
        train_labels.append(class_to_idx[_class.name])

for _class in test_data_path.iterdir():
    for img in _class.iterdir():
        _img = Image.open(img)
        grey = _img.convert("L")

        test_data.append(np.array(grey))
        test_labels.append(class_to_idx[_class.name])

train_data = np.array(train_data)
test_data = np.array(test_data)

train_labels = np.array(train_labels)
test_labels = np.array(test_labels)

In [120]:
# Shuffle the dataset 

perm = np.random.permutation(train_data.shape[0])

X_train = train_data[perm]
y_train = train_labels[perm]

X_test = test_data
y_test = test_labels

In [121]:
# Convert training data to tensors

X_train_tensor = torch.from_numpy(X_train).float().unsqueeze(1)
X_test_tensor = torch.from_numpy(X_test).float().unsqueeze(1)

y_train_tensor = torch.from_numpy(y_train).long()

In [122]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=32)

In [123]:
# Define the classes 

from enum import Enum

# Define the classes
class CIFAR10Class(Enum):
    AIRPLANE = 0
    AUTOMOBILE = 1
    BIRD = 2
    CAT = 3
    DEER = 4
    DOG = 5
    FROG = 6
    HORSE = 7
    SHIP = 8
    TRUCK = 9

In [124]:
# See the shape of the images 

X_train[0].shape

(32, 32)

In [125]:
# Build the CNN

class ImageClassifier(nn.Module):
    def __init__(self):
        super(ImageClassifier, self).__init__()

        # Convolutional layers
        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=12,
            kernel_size=5,
            padding=1
        )

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(
            in_channels=12,
            out_channels=24,
            kernel_size=5,
            padding=1
        )

        # Fully connected layers
        self.fc1 = nn.Linear(24 * 6 * 6, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        relu = F.relu
        
        x = relu(self.conv1(x))
        x = self.pool(x)

        x = relu(self.conv2(x))
        x = self.pool(x)

        x = torch.flatten(x, 1)

        x = relu(self.fc1(x))
        x = relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [126]:
# Define the model, the loss and the optimizer 

model = ImageClassifier()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001)

In [127]:
# Train the model

for _ in range(20):
    print(f"Epoch: [{_}] - ", end="")
    running_loss = 0.0
    
    for idx, data in enumerate(train_loader):
        x_batch, y_batch = data

        optimizer.zero_grad()

        yhat = model(x_batch)
        loss = criterion(yhat, y_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Loss: {running_loss / len(train_loader):.4f}")

Epoch: [0] - Loss: 1.9651
Epoch: [1] - Loss: 1.7084
Epoch: [2] - Loss: 1.5891
Epoch: [3] - Loss: 1.5008
Epoch: [4] - Loss: 1.4218
Epoch: [5] - Loss: 1.3584
Epoch: [6] - Loss: 1.3023
Epoch: [7] - Loss: 1.2532
Epoch: [8] - Loss: 1.2108
Epoch: [9] - Loss: 1.1717
Epoch: [10] - Loss: 1.1357
Epoch: [11] - Loss: 1.1018
Epoch: [12] - Loss: 1.0713
Epoch: [13] - Loss: 1.0424
Epoch: [14] - Loss: 1.0120
Epoch: [15] - Loss: 0.9861
Epoch: [16] - Loss: 0.9593
Epoch: [17] - Loss: 0.9388
Epoch: [18] - Loss: 0.9148
Epoch: [19] - Loss: 0.8902


In [130]:
# Predict from the model

model.eval()

with torch.no_grad():
    logits = model(X_test_tensor)
    y_pred = logits.argmax(dim=1).cpu().numpy()

In [131]:
# Evaluate predictions

from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.5995